# Reviewer Audit — `Exam_Pressure` Leakage / Construct-Overlap Check

This notebook reproduces the targeted audit added during peer review. It addresses whether the strong predictive contribution of `Exam_Pressure` is consistent with direct computational target leakage.

The audit includes:

1. high-stress prevalence by `Exam_Pressure` level;
2. 5-fold CV ablation: full group-blind model, `Exam_Pressure` only, and full model without `Exam_Pressure`;
3. 20 target-permutation sanity checks;
4. deterministic-label check at each observed `Exam_Pressure` value.

**Interpretation boundary:** these checks can rule against deterministic target copying and obvious pipeline leakage, but cannot prove independence from the original benchmark's undocumented label-generation mechanism.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score

DATA_CANDIDATES = [
    Path("../data/student-lifestyle-and-stress-dataset.csv"),
    Path("student-lifestyle-and-stress-dataset.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Download the Kaggle CSV and save it under ../data/.")

RESULT_DIR = Path("../outputs/reviewer_audit")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
TARGET = "Stress_Level"
NUM = ["Sleep_Hours","Study_Hours","Social_Media_Hours","Attendance","Exam_Pressure","Family_Support"]
CAT = ["Month"]
FULL = NUM + CAT
MINUS_EXAM = [c for c in FULL if c != "Exam_Pressure"]

def make_pipe(features):
    nums = [c for c in features if c in NUM]
    cats = [c for c in features if c in CAT]
    transformers = []
    if nums:
        transformers.append(("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), nums))
    if cats:
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), cats))
    return Pipeline([
        ("prep", ColumnTransformer(transformers)),
        ("lr", LogisticRegression(max_iter=2000)),
    ])


In [ ]:
# 1) Conditional prevalence and deterministic-label check
ep = (df.groupby("Exam_Pressure", dropna=False)[TARGET]
        .agg(["count","mean","sum"]).reset_index())
ep["high_stress_n"] = ep["sum"].astype(int)
ep["low_stress_n"] = ep["count"] - ep["high_stress_n"]
ep["high_stress_pct"] = 100 * ep["mean"]
ep = ep.drop(columns=["sum","mean"])
ep.to_csv(RESULT_DIR / "exam_pressure_conditional_prevalence.csv", index=False)

det = []
for val, part in df[df["Exam_Pressure"].notna()].groupby("Exam_Pressure"):
    labels = sorted(part[TARGET].unique().tolist())
    det.append({
        "Exam_Pressure": val, "n": len(part),
        "labels_present": ",".join(map(str, labels)),
        "both_classes_present": len(labels)==2,
    })
det = pd.DataFrame(det)
det.to_csv(RESULT_DIR / "exam_pressure_determinism_check.csv", index=False)
display(ep)
print("Both classes present at every observed Exam_Pressure level:", det["both_classes_present"].all())


In [ ]:
# 2) Five-fold CV ablation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
configs = {
    "Full group-blind": FULL,
    "Exam_Pressure only": ["Exam_Pressure"],
    "Full minus Exam_Pressure": MINUS_EXAM,
}
rows=[]
for name, features in configs.items():
    prob = cross_val_predict(make_pipe(features), df[features], df[TARGET],
                             cv=cv, method="predict_proba", n_jobs=-1)[:,1]
    rows.append({
        "configuration": name,
        "roc_auc": roc_auc_score(df[TARGET], prob),
        "average_precision": average_precision_score(df[TARGET], prob),
    })
ablation = pd.DataFrame(rows)
ablation.to_csv(RESULT_DIR / "exam_pressure_ablation_cv.csv", index=False)
display(ablation)


In [ ]:
# 3) Target-permutation sanity check
rng = np.random.default_rng(42)
perm_rows=[]
for i in range(20):
    y_perm = rng.permutation(df[TARGET].to_numpy())
    cvp = StratifiedKFold(n_splits=5, shuffle=True, random_state=100+i)
    aucs = cross_val_score(make_pipe(FULL), df[FULL], y_perm,
                           cv=cvp, scoring="roc_auc", n_jobs=-1)
    perm_rows.append({
        "permutation": i+1,
        "mean_roc_auc": float(np.mean(aucs)),
        "min_fold_auc": float(np.min(aucs)),
        "max_fold_auc": float(np.max(aucs)),
    })
perm = pd.DataFrame(perm_rows)
perm.to_csv(RESULT_DIR / "target_permutation_sanity.csv", index=False)
print(perm["mean_roc_auc"].describe())


In [ ]:
# 4) Figure and concise summary
fig, ax = plt.subplots(figsize=(7.5,4.5))
plot_ep = ep[ep["Exam_Pressure"].notna()].copy()
ax.plot(plot_ep["Exam_Pressure"], plot_ep["high_stress_pct"], marker="o")
ax.set_xlabel("Exam_Pressure")
ax.set_ylabel("High-stress prevalence (%)")
ax.set_title("High-stress prevalence across Exam_Pressure levels")
ax.set_ylim(0,100)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULT_DIR / "fig_exam_pressure_conditional_prevalence_600dpi.png",
            dpi=600, bbox_inches="tight")
plt.show()

summary = {
    "dataset_rows": int(len(df)),
    "full_auc": float(ablation.loc[ablation.configuration=="Full group-blind","roc_auc"].iloc[0]),
    "exam_only_auc": float(ablation.loc[ablation.configuration=="Exam_Pressure only","roc_auc"].iloc[0]),
    "minus_exam_auc": float(ablation.loc[ablation.configuration=="Full minus Exam_Pressure","roc_auc"].iloc[0]),
    "permutation_auc_mean": float(perm["mean_roc_auc"].mean()),
    "permutation_auc_min": float(perm["mean_roc_auc"].min()),
    "permutation_auc_max": float(perm["mean_roc_auc"].max()),
    "all_exam_pressure_levels_contain_both_classes": bool(det["both_classes_present"].all()),
}
(RESULT_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
